# Consolidacion de parquet por tipo

Este notebook une todos los anos por tipo desde `data/processed` y genera:

- `data/processed/fetal.parquet`
- `data/processed/nofetal.parquet`
- `data/processed/nac.parquet`


In [ ]:
from pathlib import Path
import pandas as pd

cwd = Path.cwd().resolve()
project_root = cwd.parent if cwd.name == "notebooks" else cwd
processed_dir = project_root / "data" / "processed"

# ---------------------------------------------------------------------
# 1) Consolidar por tipo
# ---------------------------------------------------------------------
type_config = {
    "fetal": {
        "input_dir": processed_dir / "fetales",
        "pattern": "fet*.parquet",
        "output_file": processed_dir / "fetal.parquet",
    },
    "nofetal": {
        "input_dir": processed_dir / "no_fetales",
        "pattern": "nofet*.parquet",
        "output_file": processed_dir / "nofetal.parquet",
    },
    "nac": {
        "input_dir": processed_dir / "nacimientos",
        "pattern": "nac*.parquet",
        "output_file": processed_dir / "nac.parquet",
    },
}

for type_name, cfg in type_config.items():
    parquet_files = sorted(cfg["input_dir"].glob(cfg["pattern"]))

    if not parquet_files:
        print(f"[WARN] No se encontraron archivos para {type_name} en {cfg['input_dir']}")
        continue

    dfs = [pd.read_parquet(file_path) for file_path in parquet_files]
    merged_df = pd.concat(dfs, ignore_index=True)

    merged_df.to_parquet(cfg["output_file"], index=False)
    print(f"[OK] {cfg['output_file'].name}: {len(merged_df):,} filas desde {len(parquet_files)} archivos")


[OK] fetal.parquet: 134,975 filas desde 5 archivos
[OK] nofetal.parquet: 1,476,123 filas desde 5 archivos
[OK] nac.parquet: 2,741,930 filas desde 5 archivos



# Agregar columnas descriptivas y limpiar columnas no requeridas


In [ ]:
DPTOS = {
    "05": "Antioquia", "08": "Atlantico", "11": "Bogota", "13": "Bolivar", "15": "Boyaca",
    "17": "Caldas", "18": "Caqueta", "19": "Cauca", "20": "Cesar", "23": "Cordoba",
    "25": "Cundinamarca", "27": "Choco", "41": "Huila", "44": "La Guajira", "47": "Magdalena",
    "50": "Meta", "52": "Narino", "54": "Norte de Santander", "63": "Quindio", "66": "Risaralda",
    "68": "Santander", "70": "Sucre", "73": "Tolima", "76": "Valle del Cauca", "81": "Arauca",
    "85": "Casanare", "86": "Putumayo", "88": "San Andres y Providencia", "91": "Amazonas",
    "94": "Guainia", "95": "Guaviare", "97": "Vaupes", "99": "Vichada",
}

AREA = {
    "1": "Cabecera municipal",
    "2": "Centro poblado",
    "3": "Rural disperso",
    "9": "Sin informacion",
}

SEXO = {"1": "Masculino", "2": "Femenino", "3": "Indeterminado"}

PESO_NAC = {
    "1": "Menos de 1.000", "2": "1.000 - 1.499", "3": "1.500 - 1.999", "4": "2.000 - 2.499",
    "5": "2.500 - 2.999", "6": "3.000 - 3.499", "7": "3.500 - 3.999", "8": "4.000 y mas", "9": "Sin informacion",
}

T_GES = {
    "1": "Menos de 22", "2": "De 22 a 27", "3": "De 28 a 37", "4": "De 38 a 41",
    "5": "De 42 y mas", "6": "Ignorado", "9": "Sin informacion",
}

EDAD_MADRE = {
    "1": "De 10-14 anos", "2": "De 15-19 anos", "3": "De 20-24 anos", "4": "De 25-29 anos", "5": "De 30-34 anos",
    "6": "De 35-39 anos", "7": "De 40-44 anos", "8": "De 45-49 anos", "9": "De 50-54 anos", "99": "Sin informacion",
}

EST_CIVM = {
    "1": "No casada, union de 2 o mas anos",
    "2": "No casada, union menor a 2 anos",
    "3": "Separada o divorciada",
    "4": "Viuda",
    "5": "Soltera",
    "6": "Casada",
    "9": "Sin informacion",
}

NIV_EDU = {
    "1": "Preescolar", "2": "Basica primaria", "3": "Basica secundaria", "4": "Media academica o clasica",
    "5": "Media tecnica", "6": "Normalista", "7": "Tecnica profesional", "8": "Tecnologica",
    "9": "Profesional", "10": "Especializacion", "11": "Maestria", "12": "Doctorado", "13": "Ninguno", "99": "Sin informacion",
}

TIPO_PARTO = {"1": "Espontaneo", "2": "Cesarea", "3": "Instrumentado", "4": "Ignorado", "9": "Sin informacion"}

SIT_PARTO = {"1": "Institucion de salud", "2": "Domicilio", "3": "Otro", "9": "Sin informacion"}

MUL_PARTO = {"1": "Simple", "2": "Doble", "3": "Triple", "4": "Cuadruple o mas", "9": "Sin informacion"}

IDHEMOCLAS = {"1": "A", "2": "B", "3": "O", "4": "AB", "9": "Sin informacion"}

IDFACTORRH = {"1": "Positivo", "2": "Negativo", "9": "Sin informacion"}

ATEN_PAR = {
    "1": "Medico", "2": "Enfermero(a)", "3": "Auxiliar de enfermeria", "4": "Promotor(a) de salud",
    "5": "Partera", "6": "Otra persona", "9": "Sin informacion",
}

TALLA_NAC = {
    "1": "Menos de 20", "2": "20-29", "3": "30-39", "4": "40-49", "5": "50-59", "6": "60 y mas", "9": "Sin informacion",
}

SEG_SOCIAL = {
    "1": "Contributivo", "2": "Subsidiado", "3": "Excepcion", "4": "Especial", "5": "No asegurado", "9": "Sin informacion",
}

IDCLASADMI = {
    "1": "EPS", "2": "EPS - Subsidiado", "3": "Entidad adaptada de salud",
    "4": "Entidad especial de salud", "5": "Entidad exceptuada de salud", "9": "Sin informacion",
}

PROFESION = {
    "1": "Medico", "2": "Enfermero(a)", "3": "Auxiliar de enfermeria", "4": "Promotor(a) de salud",
    "5": "Funcionario de registro civil", "9": "Sin informacion",
}

SIT_DEFUN = {
    "1": "Hospital/clinica", "2": "Centro/puesto de salud", "3": "Casa/domicilio",
    "4": "Lugar de trabajo", "5": "Via publica", "6": "Otro", "9": "Sin informacion",
}

MU_PARTO = {"1": "Antes", "2": "Durante", "3": "Despues", "4": "Ignorado", "9": "Sin informacion"}

TIPO_EMB = {"1": "Simple", "2": "Doble", "3": "Triple", "4": "Cuadruple o mas", "5": "Ignorado", "9": "Sin informacion"}

ASIS_MED = {"1": "Si", "2": "No", "3": "Ignorado", "9": "Sin informacion"}

EST_CIVIL = {
    "1": "No casado(a), union de 2 o mas anos",
    "2": "No casado(a), union menor a 2 anos",
    "3": "Separado(a) o divorciado(a)",
    "4": "Viudo(a)",
    "5": "Soltero(a)",
    "6": "Casado(a)",
    "9": "Sin informacion",
}

GRU_ED1 = {
    "00": "Menor de una hora", "01": "Menor de un dia", "02": "De 1 a 6 dias", "03": "De 7 a 27 dias",
    "04": "De 28 a 29 dias", "05": "De 1 a 5 meses", "06": "De 6 a 11 meses", "07": "De 1 ano",
    "08": "De 2 a 4 anos", "09": "De 5 a 9 anos", "10": "De 10 a 14 anos", "11": "De 15 a 19 anos",
    "12": "De 20 a 24 anos", "13": "De 25 a 29 anos", "14": "De 30 a 34 anos", "15": "De 35 a 39 anos",
    "16": "De 40 a 44 anos", "17": "De 45 a 49 anos", "18": "De 50 a 54 anos", "19": "De 55 a 59 anos",
    "20": "De 60 a 64 anos", "21": "De 65 a 69 anos", "22": "De 70 a 74 anos", "23": "De 75 a 79 anos",
    "24": "De 80 a 84 anos", "25": "De 85 a 89 anos", "26": "De 90 a 94 anos", "27": "De 95 a 99 anos",
    "28": "De 100 anos y mas", "29": "Edad desconocida",
}

GRU_ED2 = {
    "01": "Menor de 1 ano", "02": "De 1 a 4 anos", "03": "De 5 a 14 anos", "04": "De 15 a 44 anos",
    "05": "De 45 a 64 anos", "06": "De 65 y mas anos", "07": "Edad desconocida",
}

MUERTEPORO = {"1": "Si", "2": "No", "9": "Sin informacion"}

IDPERTET = {
    "1": "Indigena", "2": "Rom (Gitano)", "3": "Raizal de San Andres y Providencia",
    "4": "Palenquero de San Basilio", "5": "Negro(a), mulato(a), afrocolombiano(a) o afrodescendiente",
    "6": "Ninguno de las anteriores", "9": "Sin informacion",
}


def map_col(df: pd.DataFrame, source_col: str, mapping: dict, target_col: str | None = None) -> pd.DataFrame:
    if source_col not in df.columns:
        return df

    target_col = target_col or f"{source_col}_desc"
    raw = df[source_col]
    is_numeric_col = pd.api.types.is_numeric_dtype(raw)

    if source_col in {"cod_dpto", "cod_munic", "codptore", "codmunre", "gru_ed1", "gru_ed2"}:
        keys = raw.astype("string").str.zfill(2)
    else:
        if is_numeric_col:
            keys = pd.to_numeric(raw, errors="coerce").astype("Int64").astype("string")
        else:
            keys = raw.astype("string").str.strip()

    mapped = keys.map(mapping)
    mapped = mapped.where(keys.notna() & (keys != "<NA>") & (keys != ""), pd.NA)
    df[target_col] = mapped.fillna("Sin informacion")
    return df


nac_mappings = {
    "cod_dpto": DPTOS,
    "cod_munic": DPTOS,
    "areanac": AREA,
    "sit_parto": SIT_PARTO,
    "sexo": SEXO,
    "peso_nac": PESO_NAC,
    "talla_nac": TALLA_NAC,
    "aten_par": ATEN_PAR,
    "t_ges": T_GES,
    "tipo_parto": TIPO_PARTO,
    "mul_parto": MUL_PARTO,
    "idhemoclas": IDHEMOCLAS,
    "idfactorrh": IDFACTORRH,
    "edad_madre": EDAD_MADRE,
    "est_civm": EST_CIVM,
    "niv_edum": NIV_EDU,
    "codptore": DPTOS,
    "codmunre": DPTOS,
    "area_res": AREA,
    "seg_social": SEG_SOCIAL,
    "idclasadmi": IDCLASADMI,
    "niv_edup": NIV_EDU,
    "profesion": PROFESION,
}

fetal_mappings = {
    "cod_dpto": DPTOS,
    "cod_munic": DPTOS,
    "a_defun": AREA,
    "sit_defun": SIT_DEFUN,
    "sexo": SEXO,
    "codptore": DPTOS,
    "codmunre": DPTOS,
    "area_res": AREA,
    "mu_parto": MU_PARTO,
    "t_parto": TIPO_PARTO,
    "tipo_emb": TIPO_EMB,
    "t_ges": T_GES,
    "peso_nac": PESO_NAC,
    "edad_madre": EDAD_MADRE,
    "est_civm": EST_CIVM,
    "niv_edum": NIV_EDU,
    "asis_med": ASIS_MED,
}

nofetal_mappings = {
    "cod_dpto": DPTOS,
    "cod_munic": DPTOS,
    "a_defun": AREA,
    "sit_defun": SIT_DEFUN,
    "sexo": SEXO,
    "est_civil": EST_CIVIL,
    "gru_ed1": GRU_ED1,
    "gru_ed2": GRU_ED2,
    "nivel_edu": NIV_EDU,
    "muerteporo": MUERTEPORO,
    "idpertet": IDPERTET,
    "codptore": DPTOS,
    "codmunre": DPTOS,
    "area_res": AREA,
    "seg_social": SEG_SOCIAL,
    "asis_med": ASIS_MED,
}

process_plan = {
    "nac": {
        "path": processed_dir / "nac.parquet",
        "mappings": nac_mappings,
        "drop_cols": [],
    },
    "fetal": {
        "path": processed_dir / "fetal.parquet",
        "mappings": fetal_mappings,
        "drop_cols": ["tipo_defun"],
    },
    "nofetal": {
        "path": processed_dir / "nofetal.parquet",
        "mappings": nofetal_mappings,
        "drop_cols": ["tipo_defun", "ultcurfal"],
    },
}

for dataset_name, cfg in process_plan.items():
    parquet_path = cfg["path"]

    if not parquet_path.exists():
        print(f"[WARN] No existe {parquet_path}")
        continue

    df = pd.read_parquet(parquet_path)

    mapped_source_cols = []
    for source_col, mapping in cfg["mappings"].items():
        if source_col in df.columns:
            mapped_source_cols.append(source_col)
        df = map_col(df, source_col, mapping, f"{source_col}_desc")

    if "codpres" in df.columns:
        # No se incluye tabla ISO en el requerimiento, por eso se conserva codigo y se crea etiqueta basica.
        df["codpres_desc"] = df["codpres"].astype("string").fillna("Sin informacion")
        mapped_source_cols.append("codpres")

    # Eliminar columnas de codigo originales (ya mapeadas) y las columnas no requeridas.
    drop_now = [col for col in (cfg["drop_cols"] + mapped_source_cols) if col in df.columns]
    if drop_now:
        df = df.drop(columns=drop_now)

    df.to_parquet(parquet_path, index=False)
    print(f"[OK] {dataset_name}: columnas descriptivas agregadas en {parquet_path.name}")